# Dá Pra Ir? | Inteligência Artificial & Machine Learning

Global Solution FIAP 2026, tema CitizenTech.

## 1. Problema e pergunta de ML

O Dá Pra Ir? ajuda pessoas com deficiência, pessoas autistas e famílias atípicas a saber, antes de sair de casa, se um lugar vai funcionar para elas.

O recorte é o bairro Vila Clementino (subprefeitura Vila Mariana), em São Paulo, onde ficam a AACD, o Hospital do Servidor Público Estadual e o Hospital São Paulo (Unifesp).

A parte de ML responde duas perguntas com dados reais e públicos:

**Modelo A, "Quando ir?"**: regressão que prevê quantas pessoas vão entrar nas estações AACD-Servidor e Hospital São Paulo (Linha 5-Lilás) em um dia. Dia com menos gente significa menos barulho e menos multidão.

**Modelo B, "Tá funcionando?"**: classificação que prevê se um chamado de acessibilidade no SP156 (calçada, rampa, obstáculo) será resolvido em até 30 dias.

Os dois modelos usam XGBoost, com Random Forest como comparação. O treino usa dados de 2024 e o teste usa dados de 2025.

## 2. Dados (download e leitura)

Usamos três fontes públicas:

1. Metrô SP: entradas diárias por estação (compilação do pacote metrosp, a partir do Metrô SP e do Insper Dataverse).
2. SP156: chamados feitos à Prefeitura de São Paulo, 2024 e 2025.
3. INMET: dados horários da estação A701 (São Paulo, Mirante de Santana), 2024 e 2025.

Os arquivos originais ficam em `dados/brutos/` e não vão para o GitHub. Antes de baixar, o notebook confere se o arquivo já existe, para não baixar de novo a cada execução. Os dados já filtrados e pequenos são salvos em `dados/`.

In [1]:
import os
import zipfile

import numpy as np
import pandas as pd
import pyreadr
import requests

# Pastas usadas pelo notebook (caminhos relativos à pasta ml/)
PASTA_DADOS = 'dados'
PASTA_BRUTOS = os.path.join(PASTA_DADOS, 'brutos')

# Cria a pasta de dados brutos se ainda não existir
os.makedirs(PASTA_BRUTOS, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

In [2]:
def baixar(url, caminho):
    # Só baixa se o arquivo ainda não existir, para não repetir downloads grandes
    if os.path.exists(caminho):
        print('Já existe, não vou baixar de novo:', caminho)
        return
    print('Baixando:', url)
    resposta = requests.get(url, stream=True, timeout=120)
    resposta.raise_for_status()
    # Grava primeiro em um arquivo temporário: se o download cair no meio,
    # não fica um arquivo pela metade que pareceria completo na próxima execução
    temporario = caminho + '.parcial'
    with open(temporario, 'wb') as f:
        for bloco in resposta.iter_content(chunk_size=1024 * 1024):
            f.write(bloco)
    os.replace(temporario, caminho)
    print('Salvo em:', caminho)

### 2.1 Metrô SP

O pacote metrosp junta os dados do Metrô SP e do Insper Dataverse em uma tabela de entradas diárias por estação (`station_entries_daily`). O arquivo está no formato `.rds` (do R) e é lido com a biblioteca `pyreadr`.

O release `data-latest` é atualizado pelo autor do pacote de tempos em tempos, então os números podem mudar um pouco entre downloads.

In [3]:
URL_METRO = 'https://github.com/viniciusoike/metrosp/releases/download/data-latest/station_entries_daily.rds'
ARQ_METRO = os.path.join(PASTA_BRUTOS, 'station_entries_daily.rds')

baixar(URL_METRO, ARQ_METRO)

Já existe, não vou baixar de novo: dados\brutos\station_entries_daily.rds


### 2.2 SP156

Os chamados do SP156 são publicados em CSVs trimestrais grandes (mais de 100 MB cada). Baixamos os 8 trimestres de 2024 e 2025.

In [4]:
# Links dos CSVs trimestrais no portal de dados abertos da Prefeitura
BASE_SP156 = 'https://dados.prefeitura.sp.gov.br/dataset/0aecfa2b-aa3a-40d4-8183-0d4351b7fd0a/resource/'
URLS_SP156 = {
    'sp156_2024_t1.csv': BASE_SP156 + '787c054d-3e77-46c8-8713-e4e26eb2dd55/download/arquivo-final-1-trim.-2024.csv',
    'sp156_2024_t2.csv': BASE_SP156 + '2d020379-fa40-4dc0-8d17-add74b119550/download/arquivo-final-2-trim.-2024.csv',
    'sp156_2024_t3.csv': BASE_SP156 + '3c12fe0a-ec4f-4a20-a987-dd177330f4ce/download/arquivo-final-3-trim.-2024.csv',
    'sp156_2024_t4.csv': BASE_SP156 + 'ed357147-f15c-4bf0-ad15-edc963a9f130/download/arquivo-final-4-trim.-2024.csv',
    'sp156_2025_t1.csv': BASE_SP156 + '7c5aeb58-2cad-40ba-9e9e-af1c17cc5070/download/arquivo-final-1-trim-25.csv',
    'sp156_2025_t2.csv': BASE_SP156 + 'f5df7f4a-d614-4f85-b4a0-263ec9370cd9/download/arquivo-final-2-trim-25.csv',
    'sp156_2025_t3.csv': BASE_SP156 + '3b1d2648-d586-40c8-819c-24fbb8005343/download/arquivo-final-3-trim-25.csv',
    'sp156_2025_t4.csv': BASE_SP156 + 'd3a98097-8128-45d4-b8fe-21874b81c971/download/arquivo-final-4-trim-25.csv',
}

for nome, url in URLS_SP156.items():
    baixar(url, os.path.join(PASTA_BRUTOS, nome))

Já existe, não vou baixar de novo: dados\brutos\sp156_2024_t1.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2024_t2.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2024_t3.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2024_t4.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2025_t1.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2025_t2.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2025_t3.csv
Já existe, não vou baixar de novo: dados\brutos\sp156_2025_t4.csv


### 2.3 INMET

O INMET publica um ZIP por ano com todas as estações automáticas. Usamos só a estação A701 (São Paulo, Mirante de Santana).

O portal do INMET às vezes recusa downloads feitos por programa. Se isso acontecer, o notebook mostra como baixar o arquivo manualmente.

In [5]:
for ano in [2024, 2025]:
    caminho = os.path.join(PASTA_BRUTOS, f'{ano}.zip')
    try:
        baixar(f'https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip', caminho)
    except requests.RequestException as erro:
        print(f'Não foi possível baixar o arquivo do INMET de {ano}: {erro}')
        print(f'Baixe manualmente: abra https://portal.inmet.gov.br/dadoshistoricos no navegador,')
        print(f'clique em "{ano}" e salve o arquivo como ml/dados/brutos/{ano}.zip. Depois rode esta célula de novo.')

Já existe, não vou baixar de novo: dados\brutos\2024.zip
Já existe, não vou baixar de novo: dados\brutos\2025.zip


### 2.4 Leitura do Metrô

Guardamos só as duas estações do recorte, com todas as datas disponíveis. As datas anteriores a 2024 servem para calcular o movimento de 7 e 14 dias antes nos primeiros dias de 2024.

In [6]:
# O .rds tem um único objeto: a tabela de entradas diárias
metro_bruto = list(pyreadr.read_r(ARQ_METRO).values())[0]

ESTACOES = ['AACD-Servidor', 'Hospital São Paulo']
metro = metro_bruto[metro_bruto['station_name'].isin(ESTACOES)]
metro = metro[['date', 'station_name', 'value']]
metro.columns = ['data', 'estacao', 'entradas']
metro['data'] = pd.to_datetime(metro['data'])
metro = metro.sort_values(['estacao', 'data']).reset_index(drop=True)

metro.to_csv(os.path.join(PASTA_DADOS, 'metro_entradas_diarias.csv'), index=False)
metro.head()

,data,estacao,entradas
0,2018-08-31,AACD-Servidor,678.0
1,2018-09-01,AACD-Servidor,763.0
2,2018-09-02,AACD-Servidor,453.0
3,2018-09-03,AACD-Servidor,1867.0
4,2018-09-04,AACD-Servidor,2066.0


### 2.5 Leitura do SP156

Os CSVs são grandes, então lemos cada um em partes de 200 mil linhas e guardamos só os chamados de cinco serviços ligados a calçada, obstáculo e rampa (guia rebaixada).

Os cinco serviços foram escolhidos a partir de uma busca mais ampla pelas palavras calçada, rampa, rebaixamento, obstáculo e acessibilidade. Ficaram de fora serviços que não tratam de circular pela calçada, como poda de árvore, manutenção de sarjeta, pintura de guia e relógios de rua. O filtro usa o nome do serviço, porque o mesmo serviço às vezes aparece em mais de um assunto.

Colunas que **não** guardamos:
- Logradouro, Bairro, Número, CEP, Latitude e Longitude: mostram o endereço exato de quem reclamou, e o distrito já basta para o nosso recorte.
- Pesquisa de satisfação: só existe depois que o chamado é atendido, então usar essas colunas para prever se ele será atendido seria vazamento de dados. Elas não aparecem nos CSVs, só na aba "Satisfação" dos arquivos XLSX, que não usamos.

O arquivo do 2º trimestre de 2024 não tem a coluna Serviço (nem no CSV, nem na aba "Solicitações" do XLSX). Sem ela não dá para separar os serviços de acessibilidade dos outros serviços do mesmo assunto, então esse trimestre fica de fora.

In [7]:
# Colunas que guardamos e os nomes simples que usamos daqui em diante
COLUNAS_SP156 = {
    'Data de Abertura': 'data_abertura',
    'Data de Finalização': 'data_finalizacao',
    'Meio de Comunicação': 'canal',
    'Tema': 'tema',
    'Assunto': 'assunto',
    'Serviço': 'servico',
    'Status': 'status',
    'Órgão': 'orgao',
    'Prefeitura Operacional': 'subprefeitura',
    'Distrito': 'distrito',
}

# Serviços que entram no Modelo B: calçada, obstáculo e rampa (guia rebaixada)
SERVICOS_MODELO_B = [
    'Calçada pública - solicitar manutenção',
    'Acessibilidade - solicitar avaliação de obstáculo na calçada',
    'Calçada particular - denunciar calçada danificada ou inexistente',
    'Guias para travessia de pedestres - Solicitar rebaixamento',
    'Guia rebaixada - solicitar fiscalização de rebaixamento irregular',
]

# Os arquivos vêm na codificação do Windows (cp1252), não em UTF-8
CODIFICACAO = 'cp1252'

partes = []
resumo_arquivos = []
for nome in URLS_SP156:
    caminho = os.path.join(PASTA_BRUTOS, nome)
    colunas_arquivo = pd.read_csv(caminho, sep=';', encoding=CODIFICACAO, nrows=0).columns

    # Trimestre sem a coluna Serviço: só contamos os chamados e deixamos de fora
    if 'Serviço' not in colunas_arquivo:
        total = sum(p['Data de Abertura'].notna().sum()
                    for p in pd.read_csv(caminho, sep=';', encoding=CODIFICACAO, usecols=['Data de Abertura'],
                                         dtype=str, chunksize=200_000))
        resumo_arquivos.append({'arquivo': nome, 'chamados': total, 'chamados_guardados': 0,
                                'observacao': 'sem coluna Serviço, fica de fora'})
        continue

    total = 0
    guardadas = 0
    for parte in pd.read_csv(caminho, sep=';', encoding=CODIFICACAO, usecols=list(COLUNAS_SP156),
                             dtype=str, chunksize=200_000):
        # O arquivo do 1º trimestre de 2025 termina com linhas totalmente vazias
        # (só ponto e vírgula), que completam o limite de linhas do Excel. Elas não são chamados.
        parte = parte.dropna(how='all')
        total += len(parte)
        parte = parte[parte['Serviço'].str.strip().isin(SERVICOS_MODELO_B)]
        guardadas += len(parte)
        partes.append(parte)
    resumo_arquivos.append({'arquivo': nome, 'chamados': total, 'chamados_guardados': guardadas, 'observacao': ''})

sp156 = pd.concat(partes, ignore_index=True).rename(columns=COLUNAS_SP156)
sp156 = sp156[list(COLUNAS_SP156.values())]
sp156['servico'] = sp156['servico'].str.strip()

# Datas vêm como dia/mês/ano, sem hora
sp156['data_abertura'] = pd.to_datetime(sp156['data_abertura'], format='%d/%m/%Y', errors='coerce')
sp156['data_finalizacao'] = pd.to_datetime(sp156['data_finalizacao'], format='%d/%m/%Y', errors='coerce')

sp156.to_csv(os.path.join(PASTA_DADOS, 'sp156_acessibilidade.csv'), index=False)
pd.DataFrame(resumo_arquivos)

,arquivo,chamados,chamados_guardados,observacao
0,sp156_2024_t1.csv,611565,4351,
1,sp156_2024_t2.csv,511779,0,"sem coluna Serviço, fica de fora"
2,sp156_2024_t3.csv,466543,3388,
3,sp156_2024_t4.csv,576794,3187,
4,sp156_2025_t1.csv,742904,4372,
5,sp156_2025_t2.csv,589980,3777,
6,sp156_2025_t3.csv,501685,3667,
7,sp156_2025_t4.csv,479701,3648,


### 2.6 Leitura do INMET

O INMET registra as medições por hora, em horário UTC. Convertemos para o horário de Brasília (UTC menos 3 horas) para que o dia do clima seja o mesmo dia das entradas no metrô. Depois agregamos por dia: chuva total, temperatura máxima e umidade média.

O valor -9999 indica hora sem medição e vira vazio. Com a conversão de fuso, o dia 31/12/2025 fica sem as 3 últimas horas, que estariam no arquivo de 2026.

In [8]:
COLUNAS_INMET = {
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'chuva_mm',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_c',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_pct',
}

# Cada ZIP tem um CSV por estação; pegamos só o da A701
partes = []
for ano in [2024, 2025]:
    with zipfile.ZipFile(os.path.join(PASTA_BRUTOS, f'{ano}.zip')) as z:
        nome_a701 = [n for n in z.namelist() if 'A701' in n][0]
        with z.open(nome_a701) as f:
            # As 8 primeiras linhas descrevem a estação; a tabela começa na 9ª
            partes.append(pd.read_csv(f, sep=';', skiprows=8, encoding='latin1', decimal=','))
inmet_horario = pd.concat(partes, ignore_index=True)

inmet_horario = inmet_horario.rename(columns=COLUNAS_INMET)
inmet_horario = inmet_horario[['Data', 'Hora UTC'] + list(COLUNAS_INMET.values())]
inmet_horario = inmet_horario.replace(-9999, np.nan)

# Converte a hora UTC para o horário de Brasília
inmet_horario['data_hora_utc'] = pd.to_datetime(inmet_horario['Data'] + ' ' + inmet_horario['Hora UTC'].str[:4],
                                                format='%Y/%m/%d %H%M')
inmet_horario['data_hora_brasilia'] = inmet_horario['data_hora_utc'] - pd.Timedelta(hours=3)
inmet_horario['data'] = inmet_horario['data_hora_brasilia'].dt.normalize()

inmet = inmet_horario.groupby('data').agg(
    chuva_mm=('chuva_mm', 'sum'),
    temp_max_c=('temp_max_c', 'max'),
    umidade_pct=('umidade_pct', 'mean'),
    horas_com_dado=('chuva_mm', 'count'),
).reset_index()
inmet['umidade_pct'] = inmet['umidade_pct'].round(1)

# 31/12/2023 só aparece por causa da conversão de fuso (3 horas), então sai
inmet = inmet[inmet['data'] >= '2024-01-01'].reset_index(drop=True)

inmet.to_csv(os.path.join(PASTA_DADOS, 'inmet_a701_diario.csv'), index=False)
inmet.head()

,data,chuva_mm,temp_max_c,umidade_pct,horas_com_dado
0,2024-01-01,0.0,26.5,67.1,24
1,2024-01-02,0.2,30.9,67.7,24
2,2024-01-03,9.8,31.5,74.2,24
3,2024-01-04,11.8,25.6,84.1,24
4,2024-01-05,0.0,29.2,72.7,24


## 3. Validação e análise exploratória

Antes de modelar, conferimos o tamanho de cada tabela, o período coberto e quanto falta de dado.

In [9]:
def resumo(tabela, coluna_data):
    # Mostra tamanho, colunas, período e porcentagem de vazios de uma tabela
    print('Linhas e colunas:', tabela.shape)
    print('Colunas:', list(tabela.columns))
    print('Primeira data:', tabela[coluna_data].min().date(), '| Última data:', tabela[coluna_data].max().date())
    print('Porcentagem de vazios por coluna:')
    print((tabela.isna().mean() * 100).round(2).to_string())

### 3.1 Metrô

In [10]:
print('Tabela completa do metrosp (todas as estações):', metro_bruto.shape)
print('Colunas originais:', list(metro_bruto.columns))
print()
resumo(metro, 'data')

Tabela completa do metrosp (todas as estações): (248200, 9)
Colunas originais: ['date', 'year', 'line_number', 'station_id', 'station_name', 'station_code', 'line_name', 'line_name_pt', 'value']

Linhas e colunas: (5540, 3)
Colunas: ['data', 'estacao', 'entradas']
Primeira data: 2018-08-31 | Última data: 2026-04-15
Porcentagem de vazios por coluna:
data        0.0
estacao     0.0
entradas    0.0


In [11]:
# Dias com e sem dado em 2024 e 2025 para cada estação
linhas = []
for estacao in ESTACOES:
    for ano in [2024, 2025]:
        dias_do_ano = pd.date_range(f'{ano}-01-01', f'{ano}-12-31')
        tabela = metro[(metro['estacao'] == estacao) & (metro['data'].dt.year == ano)]
        com_dado = tabela.dropna(subset=['entradas'])['data']
        linhas.append({
            'estacao': estacao,
            'ano': ano,
            'dias_no_ano': len(dias_do_ano),
            'dias_com_dado': com_dado.nunique(),
            'dias_sem_dado': len(dias_do_ano.difference(com_dado)),
            'datas_repetidas': tabela['data'].duplicated().sum(),
        })
pd.DataFrame(linhas)

,estacao,ano,dias_no_ano,dias_com_dado,dias_sem_dado,datas_repetidas
0,AACD-Servidor,2024,366,366,0,0
1,AACD-Servidor,2025,365,365,0,0
2,Hospital São Paulo,2024,366,366,0,0
3,Hospital São Paulo,2025,365,365,0,0


### 3.2 SP156

Primeiro o tamanho de cada arquivo e quantos chamados guardamos. Depois, quantos chamados há de cada serviço na cidade e na subprefeitura Vila Mariana.

In [12]:
print('Total de chamados nos 8 arquivos:', sum(r['chamados'] for r in resumo_arquivos))
print('Chamados guardados (5 serviços do Modelo B):', len(sp156))
print()
resumo(sp156, 'data_abertura')

Total de chamados nos 8 arquivos: 4480951
Chamados guardados (5 serviços do Modelo B): 26390

Linhas e colunas: (26390, 10)
Colunas: ['data_abertura', 'data_finalizacao', 'canal', 'tema', 'assunto', 'servico', 'status', 'orgao', 'subprefeitura', 'distrito']
Primeira data: 2024-01-01 | Última data: 2025-12-31
Porcentagem de vazios por coluna:
data_abertura        0.00
data_finalizacao    33.96
canal                0.00
tema                 0.00
assunto              0.00
servico              0.00
status               0.00
orgao                0.00
subprefeitura        0.11
distrito             0.08


In [13]:
# Chamados de cada serviço, na cidade e na subprefeitura Vila Mariana
sp156['vila_mariana'] = sp156['subprefeitura'] == 'VILA MARIANA'
servicos = (sp156.groupby(['assunto', 'servico'])
            .agg(chamados_cidade=('status', 'size'), chamados_vila_mariana=('vila_mariana', 'sum'))
            .sort_values('chamados_cidade', ascending=False)
            .reset_index())
sp156 = sp156.drop(columns='vila_mariana')
servicos

,assunto,servico,chamados_cidade,chamados_vila_mariana
0,"Calçadas, guias e postes",Calçada pública - solicitar manutenção,9885,445
1,"Calçadas, guias e postes",Acessibilidade - solicitar avaliação de obstáculo na calçada,7086,398
2,"Calçadas, guias e postes",Calçada particular - denunciar calçada danificada ou inexistente,6764,544
3,Guias rebaixadas,Guias para travessia de pedestres - Solicitar rebaixamento,1152,85
4,Guias rebaixadas,Guia rebaixada - solicitar fiscalização de rebaixamento irregular,763,34
5,Fiação e Postes,Calçada particular - denunciar calçada danificada ou inexistente,335,19
6,Fiação e Postes,Acessibilidade - solicitar avaliação de obstáculo na calçada,303,8
7,Fiação e Postes,Calçada pública - solicitar manutenção,101,2
8,Acessibilidade -> Guias rebaixadas,Guias para travessia de pedestres - Solicitar rebaixamento,1,0


Antes de definir o alvo do Modelo B, olhamos os status e as datas de finalização.

In [14]:
print('Status dos chamados:')
print(sp156['status'].value_counts(dropna=False).to_string())
print()
print('Última data de finalização encontrada:', sp156['data_finalizacao'].max().date())
print('Chamados com finalização antes da abertura:', (sp156['data_finalizacao'] < sp156['data_abertura']).sum())
print()
print('Porcentagem de chamados com data de finalização, por status:')
print((sp156.groupby('status')['data_finalizacao'].apply(lambda s: s.notna().mean() * 100)).round(1).to_string())

Status dos chamados:
status
Finalizada      17239
Aberto           8957
Cancelada         193
Em Andamento        1

Última data de finalização encontrada: 2026-08-07
Chamados com finalização antes da abertura: 0

Porcentagem de chamados com data de finalização, por status:
status
Aberto            0.0
Cancelada       100.0
Em Andamento      0.0
Finalizada      100.0


In [15]:
# Por mês de abertura: quantos chamados e quantos já têm data de finalização
por_mes = sp156.groupby(sp156['data_abertura'].dt.to_period('M')).agg(
    chamados=('status', 'size'),
    com_finalizacao=('data_finalizacao', 'count'),
)
por_mes['pct_com_finalizacao'] = (por_mes['com_finalizacao'] / por_mes['chamados'] * 100).round(1)
por_mes

,chamados,com_finalizacao,pct_com_finalizacao
data_abertura,,,
2024-01,1396,1044,74.8
2024-02,1489,1110,74.5
2024-03,1466,1085,74.0
2024-07,1123,819,72.9
2024-08,1160,829,71.5
2024-09,1105,735,66.5
2024-10,1134,807,71.2
2024-11,1159,815,70.3
2024-12,894,601,67.2


### 3.3 Alvo do Modelo B: resolvido em até 30 dias

- **1 (resolvido):** status Finalizada e no máximo 30 dias entre a abertura e a finalização.
- **0 (não resolvido):** status Finalizada com mais de 30 dias, ou status Aberto ou Em Andamento.
- **Fora do modelo:** status Cancelada, porque quem cancelou foi o próprio cidadão.

O status foi registrado em agosto de 2026 e o último chamado foi aberto em dezembro de 2025, então todos os chamados tiveram pelo menos 7 meses para ser fechados. Quem ainda está aberto certamente não foi resolvido em 30 dias. Por isso não precisamos tirar os últimos 30 dias do período.

"Finalizada" quer dizer que a Prefeitura encerrou o chamado, não necessariamente que a calçada foi consertada.

In [16]:
# Confere se só existem os quatro status esperados antes de aplicar a regra
print('Status encontrados:', sorted(sp156['status'].unique()))

sp156_modelo = sp156[sp156['status'] != 'Cancelada'].copy()
dias_ate_finalizar = (sp156_modelo['data_finalizacao'] - sp156_modelo['data_abertura']).dt.days
sp156_modelo['resolvido_30d'] = ((sp156_modelo['status'] == 'Finalizada') & (dias_ate_finalizar <= 30)).astype(int)

print('Chamados cancelados, fora do modelo:', (sp156['status'] == 'Cancelada').sum())
print('Chamados no modelo:', len(sp156_modelo))

Status encontrados: ['Aberto', 'Cancelada', 'Em Andamento', 'Finalizada']
Chamados cancelados, fora do modelo: 193
Chamados no modelo: 26197


In [17]:
# Quantos chamados em cada classe, no treino (2024) e no teste (2025)
sp156_modelo['ano'] = sp156_modelo['data_abertura'].dt.year
classes = pd.crosstab(sp156_modelo['ano'], sp156_modelo['resolvido_30d'], margins=True, margins_name='total')
classes.columns = ['nao_resolvido (0)', 'resolvido (1)', 'total']
classes['pct_resolvido'] = (classes['resolvido (1)'] / classes['total'] * 100).round(1)
classes

,nao_resolvido (0),resolvido (1),total,pct_resolvido
ano,,,,
2024,7457,3394,10851,31.3
2025,10503,4843,15346,31.6
total,17960,8237,26197,31.4


### 3.4 INMET

In [18]:
print('Tabela horária (2024 e 2025):', inmet_horario.shape)
print()
resumo(inmet, 'data')
print()
print('Dias com menos de 24 horas de medição:', (inmet['horas_com_dado'] < 24).sum())
print(inmet.describe().round(1).to_string())

Tabela horária (2024 e 2025): (17544, 8)

Linhas e colunas: (731, 5)
Colunas: ['data', 'chuva_mm', 'temp_max_c', 'umidade_pct', 'horas_com_dado']
Primeira data: 2024-01-01 | Última data: 2025-12-31
Porcentagem de vazios por coluna:
data              0.0
chuva_mm          0.0
temp_max_c        0.0
umidade_pct       0.0
horas_com_dado    0.0

Dias com menos de 24 horas de medição: 45
                      data  chuva_mm  temp_max_c  umidade_pct  horas_com_dado
count                  731     731.0       731.0        731.0           731.0
mean   2024-12-31 00:00:00       4.1        27.3         67.4            23.9
min    2024-01-01 00:00:00       0.0        13.0         28.3             9.0
25%    2024-07-01 12:00:00       0.0        24.4         60.8            24.0
50%    2024-12-31 00:00:00       0.0        27.6         68.8            24.0
75%    2025-07-01 12:00:00       1.2        30.8         75.2            24.0
max    2025-12-31 00:00:00     125.6        37.2         94.1        